# Taller de la semana 3 — SOLUCIÓN DE REFERENCIA (solo para el docente)

**Enunciado:** [`README.md`](README.md) · **Plan de clase:** [`../../weeks/week-03.md`](../../weeks/week-03.md) · **Motor proporcionado:** [`search.py`](search.py)

Este notebook presenta la **explicación didáctica** del taller de la semana 3.
**Importa el código de referencia de [`solution.py`](solution.py)** sin volver a copiarlo,
por lo que la solución mostrada es exactamente la que evalúan las pruebas proporcionadas.
La garantía de corrección se comprueba por separado: el comando documentado
`python3 studios/_verify_solutions.py week-03` ejecuta `test_search.py` sin modificaciones
contra `solution.py`.

> No distribuir. Se excluye de la versión para estudiantes mediante `studios/.gitignore`.


In [1]:
# --- Preparación: añade la carpeta de la semana (solution/search) y la raíz del repositorio
# (aicourse) a la ruta de importación para ejecutar desde las ubicaciones previstas. ---
import sys, pathlib
# POO: pathlib.Path es una clase; cwd() devuelve un objeto con la ruta del directorio actual.
# / combina rutas y exists() comprueba la existencia del archivo mediante métodos del objeto.
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-03"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        # Método de list: añade estas carpetas al inicio de la búsqueda de módulos de Python.
        sys.path.insert(0, p)

import inspect
import solution
from search import EightPuzzle, TinyGraph, Problem, D8, load_instances

## Tarea 1 — Formula misioneros y caníbales: los cinco componentes en código

La formulación es una *decisión de diseño* y se evalúa. La solución de referencia usa
`(m_left, c_left, boat_side)` como estado, `(3,3,0)` como inicio y `(0,0,1)` como objetivo.
La restricción de seguridad de los misioneros en ambas orillas se comprueba en `actions()`:
solo son alcanzables los estados legales, por lo que `result()` conserva una transición pura.
Comprobarlo en `result()` también es defendible; en la demostración los estudiantes justifican su elección.
No hay una prueba proporcionada para este problema, así que lo comprobamos aquí:
debe resolverse con los **11** cruces clásicos del bote.


In [2]:
# getsource muestra la clase importada de solution.py sin mantener una copia del código aquí.
print(inspect.getsource(solution.MissionariesAndCannibals))

# Instanciación: p es un objeto de la subclase; su inicializador reutiliza el de Problem con super().
p = solution.MissionariesAndCannibals()
# BFS llama a la interfaz del objeto problema; devuelve un Node objetivo y un contador.
node, exp = solution.bfs(p)
assert node is not None, "M&C should be solvable"
# path() es un método de Node: sigue sus enlaces parent y reconstruye las acciones del plan.
assert len(node.path()) == 11, f"classic optimal is 11 trips, got {len(node.path())}"
# UCS, con costos uniformes aquí, coincide en la longitud:
node_u, _ = solution.ucs(p)
assert len(node_u.path()) == 11
print(f"M&C solved: {len(node.path())} trips, {exp} expansions")
print("plan:", node.path())

class MissionariesAndCannibals(Problem):
    def __init__(self):
        # Inicializa initial y goal del padre en ESTE objeto; no crea otro objeto Problem.
        super().__init__((3, 3, 0), (0, 0, 1))

    # El prefijo _ señala un método auxiliar interno por convención, no una prohibición de acceso.
    def _legal(self, state):
        """Ninguna orilla puede tener más caníbales que misioneros si hay al menos
        un misionero presente. También exige poblaciones entre 0 y 3."""
        m_left, c_left, _ = state
        if not (0 <= m_left <= 3 and 0 <= c_left <= 3):
            return False
        # La población total es fija; basta guardar la orilla izquierda y deducir la derecha.
        m_right, c_right = 3 - m_left, 3 - c_left
        if m_left > 0 and c_left > m_left:
            return False
        if m_right > 0 and c_right > m_right:
            return False
        return True

    def actions(self, state):
        m_left, c_left, boat = state
        legal = []
       

## Tarea 2 — Cuatro algoritmos, un bucle de búsqueda

BFS, DFS y UCS usan el *mismo* bucle `search()` con distinta disciplina de frontera:
`fifo`, `lifo` o `priority`; cada uno requiere una línea. IDS es el interesante:
**no** llama a `search()`, sino que ejecuta DFS recursiva con profundidad limitada y aumenta el límite.
Obtiene la completitud de BFS con memoria `O(bd)` de DFS, repitiendo el trabajo de los niveles superficiales.


In [3]:
# Las funciones también son objetos: fn guarda cada referencia y getsource obtiene su código.
for fn in (solution.bfs, solution.dfs, solution.ucs,
           solution.depth_limited, solution.ids):
    print(inspect.getsource(fn))

def bfs(problem):
    """Búsqueda en anchura: completa; óptima solo cuando los costos por paso son uniformes."""
    # FIFO retira primero el nodo más antiguo: explora por niveles, no por costo acumulado.
    return search(problem, "fifo")

def dfs(problem):
    """Búsqueda en profundidad sobre un grafo. El conjunto de explorados de
    ``search()`` evita ciclos indefinidos. No garantiza optimalidad."""
    # LIFO retira el último nodo insertado y profundiza en esa rama.
    return search(problem, "lifo")

def ucs(problem):
    """Búsqueda de costo uniforme: óptima con costos no negativos por paso."""
    # La cola de prioridad de Frontier usa node.g: retira primero el camino de menor costo.
    return search(problem, "priority")

def depth_limited(problem, limit):
    """Búsqueda recursiva en árbol hasta una profundidad fija.

    Devuelve ``(node, expansions, hit_limit)``: ``node`` es el Node objetivo
    o None; ``expansions`` cuenta nodos expandidos; ``hit_limit`` es True
    si se

### Las garantías que deben CUMPLIRSE

- `test_bfs_optimal_uniform_cost` — cada movimiento del rompecabezas cuesta 1, así que BFS *sí es* óptima (longitud 8).
- `test_ucs_optimal_nonuniform_cost` — UCS devuelve el camino realmente óptimo de costo 2 en `TinyGraph`. Es *la razón por la que existe UCS*.
- `test_dfs_terminates_on_cyclic_graph` — el conjunto de explorados de `search()` permite que DFS termine sin recorrer indefinidamente estados repetidos.
- `test_ids_matches_bfs_solution_length` — IDS encuentra una solución de longitud 8, igual que BFS, con memoria `O(bd)`.


In [4]:
D8_OPTIMAL_LEN = 8

# BFS óptima con costos uniformes
bnode, _ = solution.bfs(EightPuzzle(D8))
assert len(bnode.path()) == D8_OPTIMAL_LEN
print(f"BFS optimal on uniform costs (len={D8_OPTIMAL_LEN})")

# UCS óptima con costos no uniformes
unode, _ = solution.ucs(TinyGraph("start", "A"))
assert unode.g == 2
print("UCS optimal on non-uniform costs (g=2)")

# DFS termina en un grafo cíclico; el motor controla estados repetidos
# Herencia y polimorfismo: redefine actions/result, pero conserva el inicializador,
# is_goal y step_cost de Problem. El motor usa la misma interfaz que con EightPuzzle.
class CyclicGraph(Problem):
# EDGES es un atributo de clase compartido; initial y goal son atributos de cada instancia.
    EDGES = {"A": ["B"], "B": ["C"], "C": ["A", "G"], "G": []}
    def actions(self, state): return self.EDGES[state]
    def result(self, state, action): return action
dnode, _ = solution.dfs(CyclicGraph("A", "G"))
assert dnode is not None and dnode.state == "G"
print("DFS terminates on a cyclic graph (explored set does its job)")

# IDS coincide con la longitud de BFS usando memoria O(bd)
# Crea un problema equivalente; IDS acumula las expansiones de todos los límites que probó.
inode, iexp = solution.ids(EightPuzzle(D8))
assert len(inode.path()) == len(bnode.path()) == D8_OPTIMAL_LEN
print(f"IDS matches BFS length ({D8_OPTIMAL_LEN}); ids expansions={iexp:,}")

BFS optimal on uniform costs (len=8)
UCS optimal on non-uniform costs (g=2)
DFS terminates on a cyclic graph (explored set does its job)
IDS matches BFS length (8); ids expansions=233


## La garantía falla: BFS es subóptima con costos no uniformes

Este es **el objetivo del ejercicio** (`test_bfs_suboptimal_nonuniform_cost`).
No es un error: BFS cuenta *aristas*, no *costo*. En `TinyGraph`, el camino con menos aristas,
`start -> A`, cuesta **10**, mientras que el óptimo, `start -> B -> A`, cuesta **2**.
La optimalidad de BFS exige costos uniformes por paso; `TinyGraph` incumple esa condición deliberadamente.
Una garantía cuyo fallo no puedes observar es una garantía que no has comprendido.


In [5]:
# Dos búsquedas sobre el mismo dominio: FIFO prioriza número de pasos y UCS costo acumulado.
bfs_node, _ = solution.bfs(TinyGraph("start", "A"))
ucs_node, _ = solution.ucs(TinyGraph("start", "A"))
print(f"BFS path cost g = {bfs_node.g}  (fewest edges, ignores cost)")
print(f"UCS path cost g = {ucs_node.g}  (true optimum)")
# g es un atributo del objeto Node. Esta aserción exige el fallo de optimalidad esperado de BFS.
assert bfs_node.g == 10 and ucs_node.g == 2
print("\nGUARANTEE FAIL confirmed: BFS returned the cost-10 path though the optimum is 2 "
      "— optimality holds only under uniform step costs.")

BFS path cost g = 10  (fewest edges, ignores cost)
UCS path cost g = 2  (true optimum)

GUARANTEE FAIL confirmed: BFS returned the cost-10 path though the optimum is 2 — optimality holds only under uniform step costs.


## Tarea 3 — Medición: expansiones frente a profundidad, el entregable logarítmico

`solution.measure()` ejecuta los cuatro algoritmos sobre el banco del rompecabezas:
profundidades 4, 8, 12 y 16, con diez instancias por profundidad.
`expansions` es la unidad común de comparación (eje 3 de la ficha de evaluación).
Las líneas rectas con eje y logarítmico indican crecimiento exponencial; observarlo en tus datos
es el objetivo de la tarea 3. Aquí usamos un subconjunto para mantener rápida la ejecución
y mostramos la media de expansiones por par (profundidad, algoritmo).


In [6]:
from collections import defaultdict
bank = load_instances()
# Subconjunto: 2 instancias por profundidad para ejecutar rápido; measure() usa las 10 por defecto.
small = {d: states[:2] for d, states in bank.items()}
rows = solution.measure(bank=small)

# defaultdict es una clase de diccionario: para una clave nueva, list crea una lista vacía.
agg = defaultdict(list)
for depth, name, exp, length in rows:
    # La tupla (profundidad, nombre) agrupa resultados; append modifica la lista asociada a esa clave.
    agg[(depth, name)].append(exp)

print(f"{'depth':>6} {'algo':>5} {'mean_expansions':>16}")
for (depth, name) in sorted(agg):
    vals = agg[(depth, name)]
    print(f"{depth:>6} {name:>5} {sum(vals)/len(vals):>16,.0f}")
# Comprobación esperada: BFS, UCS e IDS devuelven la longitud óptima en cada profundidad.
# Esta comprensión guarda una longitud por par (d, n); si hay varias filas, conserva la última.
# Las aserciones siguientes comprueban solo BFS para esa última fila de cada profundidad.
by = {(d, n): L for d, n, e, L in rows}
for d in small:
    assert by[(d, "BFS")] == d, (d, by[(d,'BFS')])
print("\n(BFS solution length matches the verified optimal depth at every depth)")

 depth  algo  mean_expansions
     4   BFS               26
     4   DFS               35
     4   IDS               24
     4   UCS               26
     8   BFS              196
     8   DFS            4,576
     8   IDS              262
     8   UCS              196
    12   BFS            1,708
    12   DFS           15,709
    12   IDS            3,213
    12   UCS            1,708
    16   BFS           11,207
    16   DFS           42,924
    16   IDS           30,019
    16   UCS           11,207

(BFS solution length matches the verified optimal depth at every depth)
